In [32]:

import pandas as pd
import numpy as np
import re
import string

from bs4 import BeautifulSoup

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\AD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\AD\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:

# Load data
df = pd.read_csv("spam.csv", encoding='latin-1')

In [ ]:
print(df.shape)

print(df.isnull().sum()) # Kiểm tra số lượng giá trị thiếu trong mỗi cột

print("Duplicate rows:", df.duplicated().sum()) # Kiểm tra số lượng dòng trùng lặp

(5572, 5)
v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64
Duplicate rows: 403


In [35]:
# Chỉ giữ 2 cột chính
df = df[['v1', 'v2']]

# Xóa duplicate
df.drop_duplicates(inplace=True)

print(df.shape)

(5169, 2)


In [ ]:
# label có cân bằng không 
print(df["v1"].value_counts())

v1
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
# Tạo danh sách stopwords
stop_words = set(stopwords.words('english')) # Từ điển các từ dừng

# Function làm sạch văn bản
def clean_text(text):
    
    text = text.lower()
    # Xóa HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Xóa URL
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Xóa dấu câu
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )
    # Xóa số
    text = re.sub(r'\d+', ' ', text)
    # Tách từ
    words = text.split()
    # Xóa stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]
    # Ghép lại thành câu
    return " ".join(words)

# Apply cleaning trực tiếp vào cột v2
df["v2"] = df["v2"].apply(clean_text)

# Remove blank rows
df = df[df["v2"].str.strip() != ""]

# Remove duplicates
df = df.drop_duplicates(subset=["v2"])

# Show results
print(df.head())

# Save cleaned dataset
df.to_csv(
    "spam_cleaned.csv",
    index=False
)

print("Dataset cleaned successfully!")


     v1                                                 v2
0   ham  go jurong point crazy available bugis n great ...
1   ham                            ok lar joking wif u oni
2  spam  free entry wkly comp win fa cup final tkts st ...
3   ham                u dun say early hor u c already say
4   ham        nah dont think goes usf lives around though
Dataset cleaned successfully!


In [ ]:
saved_df = pd.read_csv("spam_cleaned.csv") # Đọc lại file đã lưu để kiểm tra

print(saved_df.shape)

(5063, 2)


In [ ]:

# chuyển văn bản thành vector số để máy tính có thể hiểu được
from sklearn.feature_extraction.text import TfidfVectorizer

# X = input text
X = df["v2"]

# y = label
y = df["v1"]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

# Convert text -> numerical vectors
X = vectorizer.fit_transform(X)

# Check matrix shape
print("TF-IDF Shape:", X.shape)

# Show some feature names
print(vectorizer.get_feature_names_out()[:20])



TF-IDF Shape: (5063, 5000)
['aah' 'aaniye' 'aaooooright' 'aathilove' 'aathiwhere' 'ab' 'abbey'
 'abdomen' 'abeg' 'abel' 'aberdeen' 'abi' 'ability' 'abiola' 'abj' 'able'
 'abnormally' 'aboutas' 'abroad' 'absence']


In [ ]:
# chia tập dữ liệu thành train và test
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

import joblib

# Save train data
joblib.dump(X_train, "X_train.pkl")
joblib.dump(y_train, "y_train.pkl")

# Save test data
joblib.dump(X_test, "X_test.pkl")
joblib.dump(y_test, "y_test.pkl")

# Save TF-IDF vectorizer
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("All files saved successfully!")

X_train shape: (4050, 5000)
X_test shape: (1013, 5000)
y_train shape: (4050,)
y_test shape: (1013,)
All files saved successfully!


In [ ]:

# Kiểm tra số lượng dòng trống
blank_rows = df["v2"].str.strip().eq("").sum()

print("Blank rows:", blank_rows)

Blank rows: 0
